## Lesson Overview

**What this lesson teaches:** how extended thinking appears in the Anthropic API, why some thinking is returned as a redacted block, and how to preserve API response blocks safely in a conversation.

**What's happening under the hood:**
1. Enable extended thinking and assign it a token budget.
2. Send Anthropic's special test string to deliberately produce redacted thinking.
3. Inspect the response as typed content blocks.
4. Extract only user-visible text for display.
5. Keep the complete assistant content when continuing the conversation.

**By the end:** you will be able to recognize and correctly handle `thinking`, `redacted_thinking`, and `text` blocks without attempting to expose private reasoning.

# Lesson 15: Handling Redacted Thinking

Extended thinking lets Claude spend additional tokens working through a problem before producing its visible answer. The API may encrypt part of that reasoning and return it as a `redacted_thinking` block. Applications should treat that block as opaque data: inspect its type, preserve it when required, and never try to decode or display it.

## The Response Shape

```text
API response
├── thinking block           → model reasoning returned by the API
├── redacted_thinking block  → opaque encrypted reasoning data
└── text block               → user-visible answer
```

A response can contain more than one block and the mix can vary. Code should branch on each block's `type` instead of assuming that `message.content[0]` contains displayable text.

## Setup

Install the dependencies once if needed:

```python
%pip install anthropic python-dotenv
```

Add `ANTHROPIC_API_KEY=...` to a `.env` file. The live example below sends an API request and may incur a small charge.

In [1]:
import os

from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = Anthropic(api_key=api_key) if api_key else None
model = 'claude-sonnet-4-5'

if client is None:
    print('Setup incomplete: configure ANTHROPIC_API_KEY.')
else:
    print(f'Anthropic client ready; model: {model}')

Anthropic client ready; model: claude-sonnet-4-5


## Conversation Helpers

The message helpers accept either plain content or an Anthropic `Message`. When an assistant response is added back to the history, its entire `content` list is retained. This matters because thinking and redacted-thinking blocks carry API state that should remain unchanged.

The `chat` helper adds the `thinking` parameter only when requested. Extended thinking uses a token budget within the response's overall `max_tokens` limit.

In [2]:
def add_user_message(messages, message):
    content = message.content if isinstance(message, Message) else message
    messages.append({'role': 'user', 'content': content})


def add_assistant_message(messages, message):
    content = message.content if isinstance(message, Message) else message
    messages.append({'role': 'assistant', 'content': content})


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    if client is None:
        raise RuntimeError('Configure ANTHROPIC_API_KEY before calling chat().')
    if thinking and thinking_budget < 1024:
        raise ValueError('thinking_budget must be at least 1024 tokens')

    params = {
        'model': model,
        'max_tokens': 4000,
        'messages': messages,
        'temperature': temperature,
        'stop_sequences': stop_sequences or [],
    }
    if thinking:
        params['thinking'] = {
            'type': 'enabled',
            'budget_tokens': thinking_budget,
        }
    if tools:
        params['tools'] = tools
    if system:
        params['system'] = system

    return client.messages.create(**params)


def text_from_message(message):
    return '\n'.join(
        block.text for block in message.content if block.type == 'text'
    )

## Trigger a Redacted-Thinking Block

Anthropic provides the following magic string specifically for testing this response path. It is not a prompt-injection technique and does not reveal reasoning; it asks the API to return a synthetic `redacted_thinking` block so an application can verify its handling logic.

### Experiment here

In the code cell directly below, edit only these two areas:

1. **The prompt passed to `add_user_message`**. Keep `thinking_test_str` to trigger a synthetic redacted block, or replace it with your own reasoning prompt:

```python
add_user_message(messages, 'Explain step by step which is larger: 9.11 or 9.9')
```

2. **The thinking settings in the `chat` call**:

```python
response = chat(messages, thinking=True, thinking_budget=1024)
```

Try `thinking=False`, or try budgets such as `2048` and `3000`. The budget must be at least `1024` when thinking is enabled and must remain below the helper's `max_tokens=4000`. This is a live API call and may incur a small charge. Rerun the inspection cell afterward.

In [11]:
thinking_test_str = (
    'ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_'
    '46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB'
)

messages = []
add_user_message(messages, "explain step by step who is the best pope over the last 500 years")

response = None
if client is None:
    print('Skipping the live request. Complete setup, then rerun this cell.')
else:
    response = chat(messages, thinking=True, thinking_budget=2048)
    print('Response received.')

Response received.


## Inspect Types, Not Private Data

For diagnostics, print the sequence of block types. Do not print a redacted block's `data` field: it is intentionally opaque and is not useful to an end user. The visible answer is assembled only from `text` blocks.

### Experiment here

First, rerun the code cell below unchanged after every prompt or budget experiment. To build a safe diagnostic logger, add these lines inside the `else:` block, after `block_types` is created:

```python
print('Model:', response.model)
print('Stop reason:', response.stop_reason)
print('Usage:', response.usage)
```

Safe fields include metadata and `[block.type for block in response.content]`. Do **not** add code that prints a thinking block's contents or a redacted block's `data`.

In [12]:
if response is None:
    print('No response to inspect yet.')
else:
    block_types = [block.type for block in response.content]
    print('Block types:', block_types)
    print('Contains redacted thinking:', 'redacted_thinking' in block_types)
    print('\nVisible text:\n')
    print(text_from_message(response))

Block types: ['thinking', 'text']
Contains redacted thinking: False

Visible text:

# Evaluating "Best" Popes: A Step-by-Step Analysis

This is ultimately **subjective** and depends on your criteria, but here's how to think through it:

## Step 1: Define Your Criteria
What makes a pope "best"?
- **Spiritual leadership** - sanctity, prayer life
- **Theological contributions** - encyclicals, teachings
- **Church reform** - addressing corruption or problems
- **Historical impact** - influence on world events
- **Pastoral care** - connection with ordinary Catholics
- **Interfaith relations** - ecumenical efforts

## Step 2: Consider Top Candidates

### **St. John XXIII (1958-1963)**
- Called Vatican II Council (modernized the Church)
- Known for warmth and humility
- Opened dialogue with modern world

### **St. John Paul II (1978-2005)**
- Helped end Soviet communism
- Most-traveled pope, global visibility
- Youth outreach, extensive writings
- Long tenure (27 years)

### **Benedict XIV (1

## Preserve the Full Assistant Response

Displaying and storing are different operations. Display only the text, but when continuing the same API conversation, append the assistant's complete structured content—including any thinking or redacted-thinking blocks—without editing or reordering it. Then add the next user turn.

### Experiment here

In the code cell below, edit only the follow-up text in this line:

```python
add_user_message(messages, 'Briefly summarize your visible answer.')
```

For example, replace it with `'Give one practical example.'`. Do not replace `add_assistant_message(messages, response)` with extracted text—the point of this experiment is to preserve every structured response block.

To actually send the prepared follow-up, create a new code cell immediately after the cell below and run:

```python
follow_up = chat(messages, thinking=True, thinking_budget=1024)
print([block.type for block in follow_up.content])
print(text_from_message(follow_up))
```

This makes another live API call. If you rerun the preparation cell, first rerun the trigger cell so that `messages` is reset and duplicate turns are not appended.

In [16]:
if response is None:
    print('No response to add to the conversation yet.')
else:
    add_assistant_message(messages, response)
    add_user_message(messages, 'Share what the answer was missing.')
    print([message['role'] for message in messages])
    print('Assistant content preserved as structured blocks:',
          isinstance(messages[1]['content'], list))

['user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user']
Assistant content preserved as structured blocks: True


In [17]:
follow_up = chat(messages, thinking=True, thinking_budget=1024)
print([block.type for block in follow_up.content])
print(text_from_message(follow_up))

['thinking', 'text']
# What My Answer Was Missing:

## **1. A Clear Position**
I hedged rather than making an actual argument—just said "it depends" without committing

## **2. Critical Analysis**
- No mention of **failures or controversies** (Pius XII's Holocaust silence, abuse crisis responses)
- Only presented positive aspects of each candidate

## **3. Recent Popes**
- **Francis** (2013-present) - first Latin American pope, focus on poor
- **Benedict XVI** (2005-2013) - theologian, rare resignation

## **4. Broader Historical Context**
- Didn't explain what **challenges** each pope faced
- No discussion of Counter-Reformation popes (1500s-1600s)
- Limited to just 4 of ~30 popes in 500 years

## **5. Diverse Perspectives**
- Presented mainly **mainstream Catholic** view
- Didn't include how non-Catholics, victims of Church failures, or progressive vs. traditional Catholics might view these popes differently

## **6. Actual Comparison**
- Listed criteria but didn't **systematically a

## Local Sanity Checks

These checks make no API calls. A tiny stand-in message confirms that text extraction ignores non-text blocks and that assistant content remains structured when it is added to history.

### Experiment here (no API cost)

In the code cell below, replace only the `fake_message = FakeMessage([...])` block and the first expected-value assertion with:

```python
fake_message = FakeMessage([
    FakeBlock('thinking'),
    FakeBlock('text', 'First visible part.'),
    FakeBlock('redacted_thinking'),
    FakeBlock('text', 'Second visible part.'),
])
assert text_from_message(fake_message) == (
    'First visible part.\nSecond visible part.'
)
```

Also update the second assertion's expected type list to `['thinking', 'text', 'redacted_thinking', 'text']`. This tests that non-text blocks are ignored and multiple visible text blocks are joined in their original order.

In [6]:
class FakeBlock:
    def __init__(self, block_type, text=None):
        self.type = block_type
        self.text = text


class FakeMessage:
    def __init__(self, content):
        self.content = content


fake_message = FakeMessage([
    FakeBlock('redacted_thinking'),
    FakeBlock('text', 'Safe visible answer.'),
])
assert text_from_message(fake_message) == 'Safe visible answer.'
assert [block.type for block in fake_message.content] == [
    'redacted_thinking', 'text'
]
print('All local checks passed.')

All local checks passed.


## Practice: Make the Handler Robust

Try these one at a time and predict the result before running. The section named beside each item tells you exactly which code cell to edit:

1. **Inspect normal thinking — edit ‘Trigger a Redacted-Thinking Block’:** replace the argument to `add_user_message` with a multi-step reasoning problem. Then run ‘Inspect Types, Not Private Data’.
2. **Change the budget — edit ‘Trigger a Redacted-Thinking Block’:** change `thinking_budget=1024` in the `chat` call to `2048` or `3000`.
3. **Multiple text blocks — edit ‘Local Sanity Checks’:** replace the `fake_message` block and its two assertions using the example above.
4. **Continue the conversation — edit ‘Preserve the Full Assistant Response’:** change the follow-up string, run that cell once, and put the provided `follow_up = chat(...)` example in a new cell beneath it.
5. **Build a safe logger — edit ‘Inspect Types, Not Private Data’:** add the three metadata `print` calls shown above while excluding thinking contents and redacted data.

## Summary

- Extended thinking is enabled with a token budget inside the response token limit.
- API responses contain typed content blocks; do not assume every block is text.
- `redacted_thinking` is opaque by design and should never be decoded or displayed.
- User interfaces should render `text` blocks and safely ignore private reasoning blocks.
- Conversation history should preserve the assistant's complete structured response unchanged.
- The official magic string tests redaction handling without revealing genuine model reasoning.